In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import cv2
import random
import shutil
!nvidia-smi
!pip install torch torchvision torchaudio
!pip install ultralytics
!pip install thop ptflops
!pip install opencv-python
!pip install codecarbon
import warnings
warnings.filterwarnings('ignore')
from codecarbon import EmissionsTracker
from ultralytics import YOLO, RTDETR
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()
import shutil
from torch.utils.data import DataLoader

/bin/bash: line 1: nvidia-smi: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.8/380.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 85.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
shutil.copytree("/kaggle/input/datasets/vaibhavdatascience/cottenweed/cottenweed",
                "/kaggle/working/data")

'/kaggle/working/data'

In [18]:
teacher = YOLO('/kaggle/input/datasets/vaibhavdatascience/1012mc-yolo11n/yolo_11n_1012_mc.yaml').load('yolo11n.pt')

teacher.train(data='/kaggle/input/datasets/vaibhavdatascience/c-yaml/cottenweed.yaml',
             epochs =25,
             imgsz=640,
             device='0,1')

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 448/499 items from pretrained weights
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/vaibhavdatascience/c-yaml/cottenweed.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=

In [22]:
student = YOLO('/kaggle/input/datasets/vaibhavdatascience/yolo11n-512mc/yolo11n_512_mc.yaml').load('yolo11n.pt')

Transferred 301/499 items from pretrained weights


In [23]:
student.train(data='/kaggle/input/datasets/vaibhavdatascience/c-yaml/cottenweed.yaml',
             epochs =1,
             imgsz=640,
             device='0,1')

Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/vaibhavdatascience/c-yaml/cottenweed.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/vaibhavdatascience/yolo11n-512mc/yolo

In [98]:
device = 'cuda'
teacher = YOLO('/kaggle/working/runs/detect/train10/weights/best.pt').model.eval().to(device)
student = YOLO("/kaggle/working/runs/detect/train12/weights/best.pt").load('yolo11n.pt').to(device)

Transferred 301/499 items from pretrained weights


In [99]:
import copy
from ultralytics.utils import DEFAULT_CFG, IterableSimpleNamespace

# 1. Load the absolute defaults (this includes box=7.5, cls=0.5, mosaic=1.0, etc.)
# We use copy to avoid modifying the global default object
hyp = copy.copy(DEFAULT_CFG)

# 2. If you want to ensure it's a SimpleNamespace (which your code was expecting)
if not isinstance(hyp, IterableSimpleNamespace):
    hyp = IterableSimpleNamespace(**hyp)

# 3. Double-check the critical ones you were missing
print(f"Verified Box Gain: {hyp.box}")      # Should be 7.5
print(f"Verified Mosaic: {hyp.mosaic}")    # Should be 1.0
print(f"Verified Mixup: {hyp.mixup}")      # Should be 0.0
print(f"Verified Mixup: {hyp.dfl}")
print(f"Verified Mixup: {hyp.cls}")

Verified Box Gain: 7.5
Verified Mosaic: 1.0
Verified Mixup: 0.0
Verified Mixup: 1.5
Verified Mixup: 0.5


In [100]:
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
from torch.utils.data import DataLoader
from types import SimpleNamespace

# 1. Load your data config (the .yaml file)
data_info = check_det_dataset('/kaggle/input/datasets/vaibhavdatascience/c-yaml/cottenweed.yaml')

hyp_namespace = hyp

# 2. Re-run the dataset initialization
train_dataset = YOLODataset(
    data=data_info,         
    img_path=data_info['train'], 
    imgsz=640,
    augment=True, 
    hyp=hyp_namespace,   # <--- Pass the Namespace, not the dict
    task='detect'
)

# 3. Create your DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    collate_fn=train_dataset.collate_fn # CRITICAL: YOLO uses a custom collate
)

Fast image access ✅ (ping: 0.0±0.0 ms, read: 1760.8±1321.5 MB/s, size: 179.5 KB)
Scanning /kaggle/working/data/train/labels.cache... 2391 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2391/2391 835.7Mit/s 0.0s
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


In [101]:
import torch
import copy
from tqdm import tqdm
from ultralytics.utils import DEFAULT_CFG
from ultralytics.utils.loss import v8DetectionLoss
import torch.nn.functional as F

# 1. Hyperparameters & Loss
hyp = copy.copy(DEFAULT_CFG)
model = student
model.to(device)
criterion = v8DetectionLoss(model.model)
criterion.hyp = hyp

# 2. Optimizer & Scheduler (Linear Decay for 50 epochs)
optimizer = torch.optim.AdamW(model.parameters(), lr=hyp.lr0, weight_decay=hyp.weight_decay)
lf = lambda x: (1 - x / hyp.epochs) * (1.0 - hyp.lrf) + hyp.lrf
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)

In [102]:
import torch.nn.functional as F

def kl_loss(s_logits, t_logits, T=3.5):
    """
    Standard KL-Divergence for logit-based distillation.
    s_logits: Student's raw class scores
    t_logits: Teacher's raw class scores
    """
    # Soften the distributions with temperature T
    p_s = F.log_softmax(s_logits / T, dim=-1)
    p_t = F.softmax(t_logits / T, dim=-1)

    # KL Divergence scaled by T^2 to keep gradients consistent
    loss = F.kl_div(p_s, p_t, reduction='batchmean') * (T**2)
    return loss/8400

In [103]:
def metric(model):
    # 3. Run validation using the official method
    # This handles the 'fuse' and 'AutoBackend' logic automatically
    metrics = model.val(data="/kaggle/input/datasets/vaibhavdatascience/c-yaml/cottenweed.yaml")

    # 4. Access your PhD metrics
    map50 = metrics.results_dict['metrics/mAP50(B)']
    map50_95 = metrics.results_dict['metrics/mAP50-95(B)']

    return map50, map50_95

In [131]:
device = 'cuda'
teacher = YOLO('/kaggle/working/runs/detect/train10/weights/best.pt').model.eval().to(device)
student = YOLO("/kaggle/working/runs/detect/train12/weights/best.pt").load('yolo11n.pt').to(device)

Transferred 301/499 items from pretrained weights


In [132]:
import torch
import copy
from tqdm import tqdm
from ultralytics.utils import DEFAULT_CFG
from ultralytics.utils.loss import v8DetectionLoss
import torch.nn.functional as F

# 1. Hyperparameters & Loss
hyp = copy.copy(DEFAULT_CFG)
model2 = student
model2.to(device)
criterion = v8DetectionLoss(model2.model)
criterion.hyp = hyp

# 2. Optimizer & Scheduler (Linear Decay for 50 epochs)
optimizer = torch.optim.AdamW(list(model2.parameters()) + list(adapters.parameters()), lr=hyp.lr0, weight_decay=hyp.weight_decay)
lf = lambda x: (1 - x / hyp.epochs) * (1.0 - hyp.lrf) + hyp.lrf
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)

In [133]:
# 1. Define 3 Adapters (Small, Medium, Large)
adapters = nn.ModuleDict({
    'p3': nn.Conv2d(64, 64, 1).to(device),
    'p4': nn.Conv2d(128, 128, 1).to(device),
    'p5': nn.Conv2d(128, 256, 1).to(device)
})

extracted_features = {}

# 2. Define the hook function
def hook_fn(name):
    def hook(module, input, output):
        extracted_features[name] = output
    return hook

# 3. Attach hooks to the specific layers (Stage P3, P4, P5)
# Indices for YOLO11: 4 (P3), 6 (P4), 10 (P5)
h1 = teacher.model[16].register_forward_hook(hook_fn('t_p3'))
h2 = teacher.model[19].register_forward_hook(hook_fn('t_p4'))
h3 = teacher.model[22].register_forward_hook(hook_fn('t_p5'))

h4 = model2.model.model[16].register_forward_hook(hook_fn('s_p3'))
h5 = model2.model.model[19].register_forward_hook(hook_fn('s_p4'))
h6 = model2.model.model[22].register_forward_hook(hook_fn('s_p5'))

In [134]:
import torch.nn.functional as F

def cwd_loss(s_feat, t_feat, T=4.0):
    # s_feat: [Batch, C, H, W]
    # t_feat: [Batch, C, H, W]
    
    B, C, H, W = s_feat.shape
    
    # Flatten spatial dimensions to [Batch, C, H*W]
    s_logits = s_feat.view(B, C, -1)
    t_logits = t_feat.view(B, C, -1)
    
    # Spatial Softmax for each channel
    s_prob = F.log_softmax(s_logits / T, dim=-1)
    t_prob = F.softmax(t_logits / T, dim=-1)
    
    # KL Divergence summed over channels
    loss = F.kl_div(s_prob, t_prob, reduction='batchmean') * (T**2)
    return loss

In [135]:
for epoch in range(100):
    model2.model.train()
    model2.model.to(device)
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{100}")
    loss_t = 0
    loss_f = 0
    
    for i, batch in pbar:
        ni = i + len(train_loader) * epoch # Integrated batch index
        for param in model2.model.parameters():
            param.requires_grad = True
        
        # --- 2. Data Preparation ---
        images = batch['img'].to(device).float() / 255
        targets = {
            'img': images,
            'bboxes': batch['bboxes'].to(device),
            'cls': batch['cls'].to(device),
            'batch_idx': batch['batch_idx'].to(device)
        }

        with torch.set_grad_enabled(True):
          # Pass raw model output, not Results objects
            s_preds = model2.model(images)
            
        loss, loss_items = criterion(s_preds, targets)

        with torch.no_grad():
            t_preds = teacher(images)

        loss_p3 = cwd_loss(extracted_features['s_p3'], extracted_features['t_p3'])
        loss_p4 = cwd_loss(extracted_features['s_p4'], extracted_features['t_p4'])
        loss_p5 = cwd_loss(adapters['p5'](extracted_features['s_p5']), extracted_features['t_p5'])

        f_loss = loss_p3 + loss_p4 + loss_p5

        total_loss = loss.sum() + 0.05 * f_loss
        
        # --- 4. Backward ---
        total_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # --- 5. Update UI ---
        pbar.set_postfix({
            'Box': f"{loss_items[0]:.3f}",
            'Cls': f"{loss_items[1]:.3f}",
            'DFL': f"{loss_items[2]:.3f}"
        })

        loss_t += total_loss
        loss_f += f_loss

    # Step the scheduler at the end of every epoch
    print(f'Total loss: {loss_t/len(train_loader)} | KL Loss: {loss_f/len(train_loader)}')   
    scheduler.step()

Epoch 1/100: 100%|██████████| 150/150 [01:40<00:00,  1.49it/s, Box=2.429, Cls=3.698, DFL=2.838]

Total loss: 156.0234832763672 | KL Loss: 77.73967742919922



Epoch 2/100: 100%|██████████| 150/150 [01:40<00:00,  1.49it/s, Box=1.941, Cls=3.297, DFL=2.453]

Total loss: 138.15469360351562 | KL Loss: 67.42284393310547



Epoch 3/100: 100%|██████████| 150/150 [01:40<00:00,  1.49it/s, Box=2.033, Cls=3.470, DFL=2.441]

...

Total loss: 42.65052032470703 | KL Loss: 50.9340705871582



Epoch 99/100: 100%|██████████| 150/150 [01:40<00:00,  1.49it/s, Box=0.642, Cls=1.104, DFL=1.053]

Total loss: 43.01859664916992 | KL Loss: 51.12516403198242



Epoch 100/100: 100%|██████████| 150/150 [01:41<00:00,  1.48it/s, Box=0.644, Cls=0.807, DFL=1.117]

Total loss: 42.867488861083984 | KL Loss: 51.17241668701172


In [138]:
metric(model2)

YOLO11n_512_mc summary (fused): 101 layers, 1,495,148 parameters, 14,836 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3035.6±942.4 MB/s, size: 171.7 KB)
val: Scanning /kaggle/working/data/valid/labels.cache... 1342 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1342/1342 255.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 84/84 1.2it/s 1:120.8sss
                   all       1342       2004       0.79      0.791      0.836      0.776
             Waterhemp        359        501       0.85       0.89      0.936      0.864
          MorningGlory        240        280        0.9      0.882      0.931      0.829
              Purslane        203        264      0.833       0.78      0.884      0.801
         SpottedSpurge         78         93      0.908      0.846      0.922      0.856
            Carpetweed        116        158      0.844      0.756      0.847      0.782
             

(0.8358327692946509, 0.7762043339057584)

In [144]:
# Iterates through every sub-module in the model and clears all forward hooks
for module in model2.model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()
    module._backward_hooks.clear()

print("All hooks cleared successfully.")

# Now try saving again
model2.save('AGRO_KD_M2_distilled.pt')

All hooks cleared successfully.


In [2]:
model = YOLO('/kaggle/working/runs/detect/train10/weights/best.pt')

In [3]:
import time

In [4]:
lat = []
for i in range(20):
    start_time = time.perf_counter()

    model('/kaggle/input/datasets/vaibhavdatascience/cottenweed/cottenweed/valid/images/180_20210628_iPhoneSE_YL_166.jpg')

    end_time = time.perf_counter()
    lat.append(end_time - start_time)
print(sum(lat)/20)


image 1/1 /kaggle/input/datasets/vaibhavdatascience/cottenweed/cottenweed/valid/images/180_20210628_iPhoneSE_YL_166.jpg: 640x480 1 PalmerAmaranth, 310.5ms
Speed: 14.1ms preprocess, 310.5ms inference, 28.7ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /kaggle/input/datasets/vaibhavdatascience/cottenweed/cottenweed/valid/images/180_20210628_iPhoneSE_YL_166.jpg: 640x480 1 PalmerAmaranth, 133.6ms
Speed: 2.8ms preprocess, 133.6ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /kaggle/input/datasets/vaibhavdatascience/cottenweed/cottenweed/valid/images/180_20210628_iPhoneSE_YL_166.jpg: 640x480 1 PalmerAmaranth, 126.2ms
Speed: 3.5ms preprocess, 126.2ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /kaggle/input/datasets/vaibhavdatascience/cottenweed/cottenweed/valid/images/180_20210628_iPhoneSE_YL_166.jpg: 640x480 1 PalmerAmaranth, 115.8ms
Speed: 2.8ms preprocess, 115.8ms inference, 1.2ms postprocess per image at shape